# 09 — Final model comparison and ranking

This notebook compares one final representative from each modelling approach while retaining every generated and provided baseline in the repository. It ranks models using the same public five-minute scorer; it does not choose a model from intuition or copy labels from the public ground truth.

The comparison is layered: most approaches share the canonical observations and scenario detector, then differ in how they resolve missing-ball intervals, transitions, probabilities, Viterbi paths, or durations. The intended architecture is `raw observations -> derived observations -> scenario evidence -> anchors -> transition/no-switch hypotheses -> interval resolver -> optional HMM/HSMM/Viterbi smoothing -> H/A/D/X labels`. A model labelled `Kalman` is therefore a Kalman-supported possession policy, not a Kalman possession classifier. The Haiqal row is deliberately outside this shared scenario layer: it is an independent raw-observation distance/speed/dead-ball HMM baseline adapted from `origin/haiqal`, so the comparison can tell whether scenario reasoning adds value. The combined Haiqal row is a separate evidence-only ablation: it sends Haiqal's raw closeness into our transition-trust layer, while our anchors, interval hypotheses, and resolver still decide ownership. George's row is another evidence-only ablation: source-aware fused ball observations are attached before feature/scenario construction, while the same resolver still decides possession. The optional Bayesian-anchor, HSMM, MLE-calibrated, and EM experiments are ranked here only after their outputs are generated and scored on the same evaluation slice.

The two latest transition experiments are also included as diagnostics: they test whether a proposed switch should be confirmed by post-contact motion plus team-level arrival, and whether a restart should remain `D` until restart-compatible movement plus stable control. They are deliberately ranked rather than silently promoted; a lower spell score means the signal is not yet safe as a direct label rewrite.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while not (ROOT / 'evaluation').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.environ['MTAH_DATA_DIR'] = str(ROOT / 'data')
RESULTS_DIR = ROOT / 'evaluation' / 'results'
SCOREBOARD_PATH = ROOT / 'evaluation' / 'scoreboard.csv'
PROVIDED_SCOREBOARD_PATH = ROOT / 'evaluation' / 'compare_results' / 'scoreboard.csv'
RERUN_MODELS = False
RANK_BY = ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
SHOW_ALL_GENERATED_RESULTS = False
print(f'RERUN_MODELS={RERUN_MODELS}')
print(f'ranking keys: {RANK_BY}')
print(f'results directory: {RESULTS_DIR}')

RERUN_MODELS=False
ranking keys: ['spell_f1_mean', 'macro_f1', 'trans_f1_25']
results directory: /Users/luke/Desktop/barca-hack-repo-staging/evaluation/results


## Re-run toggle

Leave `RERUN_MODELS = False` to load the committed results immediately. Set it to `True` when the feature or model code has changed; the scenario-aware experiment runners and the independent Haiqal runner will regenerate their outputs, then the common evaluator will rebuild the scoreboard. The runners intentionally retain their full baseline output sets; the ranking table below only presents the final representatives.

In [2]:
RUNNER_MODULES = [
    'possession_lab.run_interpolated_flight_experiment',
    'possession_lab.run_temporal_experiment',
    'possession_lab.run_haiqal_baseline',
    'possession_lab.run_calibration_experiment',
]

if RERUN_MODELS:
    runner_environment = os.environ.copy()
    runner_environment['MTAH_DATA_DIR'] = str(ROOT / 'data')
    for module in RUNNER_MODULES:
        print(f'Running {module} ...')
        completed = subprocess.run(
            [sys.executable, '-m', module],
            cwd=ROOT,
            env=runner_environment,
            capture_output=True,
            text=True,
            check=False,
        )
        print(completed.stdout[-2000:])
        if completed.returncode:
            print(completed.stderr[-4000:])
            raise RuntimeError(f'{module} failed with exit status {completed.returncode}')
    evaluator = subprocess.run(
        [sys.executable, str(ROOT / 'evaluation' / 'run_evaluation.py'), '--gt', 'public', '--no-detail'],
        cwd=ROOT,
        capture_output=True,
        text=True,
        check=False,
    )
    print(evaluator.stdout[-4000:])
    if evaluator.returncode:
        print(evaluator.stderr[-4000:])
        raise RuntimeError(f'public evaluator failed with exit status {evaluator.returncode}')
else:
    print('Loading committed model outputs; no runners were started.')

Loading committed model outputs; no runners were started.


## Final representatives

The manifest is intentionally small and auditable. The four `provided baseline` rows come from the supplied comparison set. The remaining rows are one representative of the current scenario, missing-ball, interval-transition, probability-emission, Viterbi, and HSMM approaches, plus the independent Haiqal raw-observation baseline. The visible-ball NO_SWITCH row tests the new interval-level retention alternative: it keeps the prior team when the transition ledger supports no switch and no new control anchor confirms a change. Optional Bayesian-anchor and additional duration rows are included when their runner outputs exist. Haiqal does not consume scenario evidence; its row tests the value of the layered architecture against a standalone distance/speed/dead-ball HMM. All other generated CSVs remain available for audit and can be shown with `SHOW_ALL_GENERATED_RESULTS`.

In [3]:
manifest = pd.DataFrame([
    {'submission': 'stage_hq_viterbi', 'approach': 'provided baseline', 'decision_layer': 'provided Viterbi decoder'},
    {'submission': 'evidence_bounded_deadball', 'approach': 'provided baseline', 'decision_layer': 'provided evidence-bounded deadball'},
    {'submission': 'aligned_physics_veto', 'approach': 'provided baseline', 'decision_layer': 'provided aligned physics veto'},
    {'submission': 'protected_aligned_reception', 'approach': 'provided baseline', 'decision_layer': 'provided protected reception'},
    {'submission': 'diagnostic_visible_ball_low_motion_dead_ball', 'approach': 'scenario + flight', 'decision_layer': 'visible-ball low-motion deadball candidate'},
    {'submission': 'diagnostic_visible_ball_no_switch_interval', 'approach': 'scenario + interval transition', 'decision_layer': 'visible-ball deadball + NO_SWITCH interval resolver'},
    {'submission': 'baseline_scenario_bidirectional_interpolated_flight_ensemble', 'approach': 'scenario + flight', 'decision_layer': 'bidirectional scenario/interpolated flight ensemble'},
    {'submission': 'baseline_same_team_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team trajectory bridge'},
    {'submission': 'baseline_same_team_kalman_missing_ball_trajectory', 'approach': 'missing-ball geometry', 'decision_layer': 'same-team Kalman-supported trajectory bridge'},
    {'submission': 'baseline_transition_interval_hypothesis', 'approach': 'interval transition', 'decision_layer': 'explicit transition interval hypothesis'},
    {'submission': 'baseline_trusted_transition', 'approach': 'interval transition', 'decision_layer': 'trusted transition resolver'},
    {'submission': 'baseline_transition_ledger_viterbi', 'approach': 'Viterbi', 'decision_layer': 'transition-ledger Viterbi'},
    {'submission': 'baseline_probability_transition_ledger_viterbi', 'approach': 'probability + Viterbi', 'decision_layer': 'normalized evidence-emission Viterbi'},
    {'submission': 'baseline_scenario_confidence_duration', 'approach': 'HSMM/duration', 'decision_layer': 'scenario confidence duration decoder'},
    {'submission': 'baseline_restart_type_specific_interpolated_flight', 'approach': 'restart + flight', 'decision_layer': 'restart-type-specific interpolated flight'},
    {'submission': 'baseline_haiqal_raw_viterbi', 'approach': 'independent raw HMM', 'decision_layer': 'Haiqal distance/speed/dead-ball clues + Viterbi'},
    {'submission': 'diagnostic_no_switch_true_switch_confirmation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + post-contact/team-level switch confirmation'},
    {'submission': 'diagnostic_no_switch_restart_activation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + restart-compatible control activation'},
    {'submission': 'diagnostic_no_switch_true_switch_restart_activation', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + switch confirmation + restart activation'},
    {'submission': 'diagnostic_no_switch_hysteretic_handoff', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + delayed stable-anchor handoff'},
    {'submission': 'diagnostic_no_switch_restart_state_machine', 'approach': 'transition signal diagnostic', 'decision_layer': 'NO_SWITCH + bounded restart state machine'},
    {'submission': 'diagnostic_no_switch_restart_hysteretic_handoff', 'approach': 'transition signal diagnostic', 'decision_layer': 'restart state machine + delayed stable-anchor handoff'},
    {'submission': 'diagnostic_no_switch_restart_phase_interval', 'approach': 'restart phase + interval transition', 'decision_layer': 'restart evidence -> transition constraints -> shared interval resolver'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard05', 'approach': 'local restart boundary diagnostic', 'decision_layer': '5-frame restart boundary guard'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard12', 'approach': 'local restart boundary diagnostic', 'decision_layer': '12-frame restart boundary guard'},
    {'submission': 'diagnostic_no_switch_restart_phase_boundary_guard25', 'approach': 'local restart boundary diagnostic', 'decision_layer': '25-frame restart boundary guard'},
])
optional_manifest = pd.DataFrame([
    {'submission': 'baseline_haiqal_evidence_probability_ledger_viterbi', 'approach': 'our pipeline + Haiqal evidence', 'decision_layer': 'Haiqal raw clues + our probability/transition ledger + Viterbi'},
    {'submission': 'diagnostic_haiqal_visible_ball_no_switch_interval', 'approach': 'our best + Haiqal evidence', 'decision_layer': 'Haiqal closeness observation + visible-ball D + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_visible_ball_no_switch_bootstrap_guard', 'approach': 'nonparametric stability guard', 'decision_layer': 'frame bootstrap lower bound + NO_SWITCH interval resolver'},
    {'submission': 'diagnostic_george_fused_observed_no_switch_interval', 'approach': 'our best + George observations', 'decision_layer': 'George fused observed ball gaps + visible-ball D + NO_SWITCH interval resolver'},
    {'submission': 'baseline_bayesian_hmm', 'approach': 'Bayesian HMM', 'decision_layer': 'Haiqal-style Dirichlet anchor transition prior + Viterbi'},
    {'submission': 'baseline_hmm_anchor', 'approach': 'HMM', 'decision_layer': 'anchor-initialized diagonal-Gaussian HMM'},
    {'submission': 'baseline_hsmm_anchor', 'approach': 'HSMM/duration', 'decision_layer': 'anchor-initialized duration cleanup'},
    {'submission': 'baseline_scenario_hsmm', 'approach': 'HSMM/duration', 'decision_layer': 'scenario-aware semi-Markov decoder'},
])

current_scores = pd.read_csv(SCOREBOARD_PATH).assign(score_source='current runners')
provided_scores = pd.read_csv(PROVIDED_SCOREBOARD_PATH).assign(score_source='provided comparison set')
scores = pd.concat([current_scores, provided_scores], ignore_index=True, sort=False)
scores = scores.drop_duplicates(subset=['submission'], keep='first')
manifest = pd.concat([manifest, optional_manifest[optional_manifest['submission'].isin(scores['submission'])]], ignore_index=True)
ranked = manifest.merge(scores, on='submission', how='left', validate='one_to_one')
missing = ranked.loc[ranked['macro_f1'].isna(), 'submission'].tolist()
if missing:
    raise FileNotFoundError(f'Manifest outputs are not scored: {missing}')
ranked = ranked.sort_values(RANK_BY, ascending=False).reset_index(drop=True)
ranked = ranked.drop(columns=['rank'], errors='ignore')
ranked.insert(0, 'rank', ranked.index + 1)
ranking_columns = ['rank', 'approach', 'decision_layer', 'submission', 'score_source', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D']
display(ranked[ranking_columns])

if SHOW_ALL_GENERATED_RESULTS:
    all_columns = [c for c in ['rank', 'submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25', 'f1_H', 'f1_A', 'f1_D'] if c in current_scores]
    display(current_scores.sort_values(['spell_f1_mean', 'macro_f1'], ascending=False)[all_columns])

,rank,approach,decision_layer,submission,score_source,macro_f1,spell_f1_mean,trans_f1_25,f1_H,f1_A,f1_D
0,1,scenario + interval transition,visible-ball deadball + NO_SWITCH interval res...,diagnostic_visible_ball_no_switch_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
1,2,transition signal diagnostic,NO_SWITCH + delayed stable-anchor handoff,diagnostic_no_switch_hysteretic_handoff,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
2,3,restart phase + interval transition,restart evidence -> transition constraints -> ...,diagnostic_no_switch_restart_phase_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
3,4,our best + Haiqal evidence,Haiqal closeness observation + visible-ball D ...,diagnostic_haiqal_visible_ball_no_switch_interval,current runners,0.891747,0.620690,0.457143,0.913319,0.852768,0.909156
4,5,local restart boundary diagnostic,5-frame restart boundary guard,diagnostic_no_switch_restart_phase_boundary_gu...,current runners,0.891727,0.620690,0.457143,0.913991,0.851708,0.909481
5,6,local restart boundary diagnostic,12-frame restart boundary guard,diagnostic_no_switch_restart_phase_boundary_gu...,current runners,0.891696,0.620690,0.400000,0.914934,0.850222,0.909932
6,7,local restart boundary diagnostic,25-frame restart boundary guard,diagnostic_no_switch_restart_phase_boundary_gu...,current runners,0.887296,0.620690,0.400000,0.912851,0.847451,0.901587
7,8,scenario + flight,visible-ball low-motion deadball candidate,diagnostic_visible_ball_low_motion_dead_ball,current runners,0.888744,0.600000,0.444444,0.909837,0.847238,0.909156
8,9,nonparametric stability guard,frame bootstrap lower bound + NO_SWITCH interv...,diagnostic_visible_ball_no_switch_bootstrap_guard,current runners,0.888744,0.600000,0.444444,0.909837,0.847238,0.909156
9,10,transition signal diagnostic,NO_SWITCH + bounded restart state machine,diagnostic_no_switch_restart_state_machine,current runners,0.824484,0.593103,0.352941,0.877173,0.843993,0.752286


## Haiqal evidence ablation

This compares the independent Haiqal output with two integration boundaries. The probability-ledger variant imports only Haiqal's raw distance, speed, visibility, out-of-pitch, and dead-ball clues; the best-model variant sends only the bounded closeness observation into our transition-trust layer. Neither imports Haiqal's final labels or replaces our resolver. The best-model combination changed 0 full-match frames and tied the current best exactly (`0.891747` Macro F1 / `0.620690` Spell F1 / `0.457143` Transition F1@25). That is a useful negative result: Haiqal's distance signal is already represented by our nearest-player/reception evidence, or is too weak to alter a guarded decision. It should remain an available observation adapter, but it is not promoted as an ensemble gain. The latest `origin/haiqal` commit (`5092f12`) was also run directly: its raw model produces the same public metrics as our adapter (`0.7677` / `0.5143` / `0.2667`), with only five implementation-level boundary frames differing from the adapter because of numeric/loading details.

## Transition-signal ablation

The runner also evaluates two promising, label-free signals from the disagreement audit. `true_switch_confirmation` asks for post-contact ball deceleration/direction change, team-level receiver arrival, and persistence of the new controlled state. `restart_activation` keeps `D` until restart-compatible ball movement agrees with stable team-level control. Both are composed from observations already available to the pipeline; neither reads public labels during inference. On this five-minute evaluation slice, the direct rewrites were not promoted:

| Diagnostic | Macro F1 | Spell F1 | Transition F1@25 | Interpretation |
|---|---:|---:|---:|---|
| NO_SWITCH + true-switch confirmation | 0.8866 | 0.3184 | 0.3273 | switch confirmation was too aggressive and fragmented spells |
| NO_SWITCH + restart activation | 0.8234 | 0.5600 | 0.3429 | restart boundary timing improved over some baselines but lost live-state continuity |
| NO_SWITCH + both signals | 0.8183 | 0.2880 | 0.2545 | composing direct rewrites amplified both failure modes |

The result is useful rather than a failure: these observations remain candidates for a guarded evidence layer, but the functions should not directly overwrite labels. The current best remains `diagnostic_visible_ball_no_switch_interval`; the next experiment should use the signals to raise or lower transition trust inside an interval resolver and preserve an explicit no-switch/contested alternative.

### Guarded handoff and restart-state diagnostics

A delayed handoff that requires a later stable anchor produced exactly the same output as the current best (`0.8917` / `0.6207` / `0.4571`), so it is a safe no-op rather than a gain. The bounded restart state machine improved the direct restart rewrite to `0.8245` / `0.5931` / `0.3529`, but it remains below the active model; adding delayed handoff did not change that result. These diagnostics therefore stay in the appendix.

The new `diagnostic_no_switch_restart_phase_interval` candidate changes the architecture rather than the score: it emits `PRE_RESTART`, `ACTIVE_UNCONTROLLED`, and `CONTROL_CONFIRMED` evidence, blocks an unconfirmed H/A handoff, and releases `D` only after sustained independent team control. It ties the current best exactly on the public slice (`0.8917` / `0.6207` / `0.4571`) and changes only frames outside the 7,500 graded public frames. It is therefore a structural full-match diagnostic, not a claimed metric improvement.

The local-boundary guards test the narrower alternative: constrain only the first proposed restart handoff instead of suppressing the entire phase. The 5-frame guard is the least disruptive (628 full-match frames changed) but still does not improve the active public score; 12 and 25 frames are progressively worse on transition timing. These remain diagnostic, and the original NO_SWITCH resolver remains the private baseline.

### Nonparametric transition stability

The runner now writes `transition_bootstrap_stability.csv`. For each inferred interval it resamples the underlying per-frame trust, activity, contest, dead-ball, and uncertainty evidence, reconstructs the interval scores, and records how often `H`, `A`, `NO_SWITCH`, or `CONTESTED` wins. This is an empirical stability estimate, not a calibrated probability of the hidden label. The channel report `transition_channel_stability.csv` separately leaves out anchors, ball motion/contact, player arrival, missing-ball cause, restart context, and Haiqal observations.

On this run, 13 of 1,306 intervals had bootstrap state stability below 0.80. Applying the 2.5th-percentile no-switch bound as a hard label guard changed 1,671 full-match frames and reduced the public result to Macro F1 `0.888744`, Spell F1 `0.600000`, and Transition F1@25 `0.444444`, below the current best `0.891747` / `0.620690` / `0.457143`. Therefore bootstrap is retained as an ambiguity and audit signal, not promoted as a label rewrite. The channel audit is the better next use: it identifies which evidence families are indispensable without using ground truth.

In [4]:
ablation_names = [
    'baseline_haiqal_raw_viterbi',
    'baseline_probability_transition_ledger_viterbi',
    'baseline_haiqal_evidence_probability_ledger_viterbi',
    'diagnostic_visible_ball_low_motion_dead_ball',
]
ablation_columns = [c for c in ['submission', 'macro_f1', 'spell_f1_mean', 'trans_f1_25'] if c in current_scores]
display(current_scores[current_scores['submission'].isin(ablation_names)][ablation_columns].sort_values('submission'))

,submission,macro_f1,spell_f1_mean,trans_f1_25
29,baseline_haiqal_evidence_probability_ledger_vi...,0.771958,0.567742,0.333333
86,baseline_haiqal_raw_viterbi,0.767720,0.514286,0.266667
32,baseline_probability_transition_ledger_viterbi,0.771958,0.567742,0.333333
12,diagnostic_visible_ball_low_motion_dead_ball,0.888744,0.600000,0.444444


## Calibration diagnostics (reported separately from the model ranking)

MLE temperature scaling is not a possession submission and is not ranked by macro-F1/spell-F1. It is evaluated with log loss and Brier score on a disjoint public holdout. A worse holdout score is useful evidence of overfitting or distribution shift, not a reason to hide the experiment.

In [5]:
calibration_path = ROOT / 'evaluation' / 'reports' / 'probability_calibration_public_holdout.csv'
if calibration_path.exists():
    display(pd.read_csv(calibration_path))
else:
    print('Calibration report is absent; set RERUN_MODELS=True to generate it.')

bootstrap_path = ROOT / 'evaluation' / 'reports' / 'transition_bootstrap_stability.csv'
if bootstrap_path.exists():
    bootstrap = pd.read_csv(bootstrap_path)
    display(bootstrap[['stable_state', 'stable_state_fraction', 'no_switch_evidence_lower']].describe())

channel_path = ROOT / 'evaluation' / 'reports' / 'transition_channel_stability.csv'
if channel_path.exists():
    channel = pd.read_csv(channel_path)
    display(channel.groupby('channel').agg(intervals=('interval_id', 'size'), agreement=('state_agrees', 'mean'), changed_intervals=('state_agrees', lambda values: int((~values).sum()))).reset_index())

,split,model,temperature,log_loss,brier
0,holdout,rule_evidence_softmax,1.000000,1.366923,0.699768
1,holdout,mle_temperature_scaled,0.778684,1.523411,0.741228


,stable_state_fraction,no_switch_evidence_lower
count,1306.000000,1306.000000
mean,0.996489,0.627934
std,0.029983,0.134399
min,0.558594,0.236111
25%,1.000000,0.576612
50%,1.000000,0.640945
75%,1.000000,0.733559
max,1.000000,0.877988


,channel,intervals,agreement,changed_intervals
0,anchors,1306,0.977795,29
1,ball_motion_contact,1306,0.845329,202
2,haiqal,1306,0.981623,24
3,missing_ball_cause,1306,0.981623,24
4,player_arrival,1306,0.906585,122
5,restart_context,1306,0.981623,24


## How to interpret the approaches

- Scenario detection is shared evidence for most current models; it is not something to exclude when testing transitions.
- The probability model is a normalized evidence-emission variant, not calibrated probability and not a replacement for scenario detection.
- MLE temperature scaling is supervised calibration only when fitted on a declared reviewed-label split; it must be scored on a disjoint holdout.
- Haiqal's Bayesian-anchor model uses a Dirichlet-smoothed transition prior from algorithmic anchors, not public labels; it is a useful transition-prior comparison, not calibrated softmax output.
- EM is an unsupervised latent-state experiment and must be checked for label switching, anchor violations, and stability before promotion.
- Viterbi is a path smoother over those emissions and transition hypotheses; it is not the primary truth-maker.
- HSMM adds duration assumptions; it remains a comparison approach until duration distributions are learned from reviewed labels.
- Kalman is tested as missing-ball geometry support. Its score answers whether the bridge helps the complete policy, not whether Kalman alone labels possession.
- Rankings on the five-minute public slice are evidence, not universal truth; spell and transition scores are especially noisy on this short benchmark.
- The latest signal ablation demonstrates why ranking matters: plausible observations can be valuable for explaining a transition while still being unsafe as a hard relabelling rule. Keep the evidence, reject the direct rewrite, and test it again inside the resolver.